# ReadyNow! — cleaned ADC-only Google ADK notebook
Preserves root/greeter + six-stage sequential emergency pipeline. Run cells top-to-bottom in the **current lab**. No API keys. Changes reduce unnecessary token usage, isolate tests, retry 429 with backoff, and show HTTP failures. **Retries cannot override a hard project quota.**

In [1]:
%pip install -q "google-cloud-aiplatform[agent_engines,adk]" google-adk google-genai requests google-auth

## 1. Configuration and ADC

In [2]:
import os, re, json, uuid, logging, asyncio, random
from datetime import datetime, timezone
from urllib.parse import quote
import requests
import google.auth
from google.auth.transport.requests import Request as GoogleAuthRequest
import vertexai
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-02-a7c8d231f9d6"  # Verify this is the current lab
LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"
NWS_CONTACT = "replace-with-valid-contact@example.com"  # Set a real contact before NWS requests

os.environ.update({"GOOGLE_CLOUD_PROJECT": PROJECT_ID,
                   "GOOGLE_CLOUD_LOCATION": LOCATION,
                   "GOOGLE_GENAI_USE_VERTEXAI": "TRUE"})
credentials, adc_project = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
print("Configured project:", PROJECT_ID)
print("ADC detected project:", adc_project)
print("Credential class:", type(credentials).__name__)
print("Note: ADC project metadata does not establish IAM access.")

Configured project: qwiklabs-gcp-02-a7c8d231f9d6
ADC detected project: qwiklabs-gcp-02-a7c8d231f9d6
Credential class: Credentials
Note: ADC project metadata does not establish IAM access.


## 2. HTTP tools (ADC only)
Legacy Geocoding API authentication may be rejected even with valid ADC. Errors are returned with HTTP/API status so the failure is visible. If the service does not support this authentication path in your project, it cannot be bypassed by changing agent prompts.

In [3]:
HTTP = requests.Session()
HTTP.headers.update({"User-Agent": f"ReadyNowPOC/1.0 ({NWS_CONTACT})"})
NWS_HEADERS = {"Accept": "application/geo+json", "User-Agent": f"ReadyNowPOC/1.0 ({NWS_CONTACT})"}

def _get_adc_token():
    try:
        if not credentials.valid:
            credentials.refresh(GoogleAuthRequest())
        return credentials.token
    except Exception as exc:
        return None

def _http_error(response):
    try:
        payload = response.json()
    except ValueError:
        payload = response.text[:350]
    # Never print tokens or request headers.
    return {"status": "unavailable", "http_status": response.status_code,
            "api_error": str(payload)[:700]}

def geocode_location(location: str) -> dict:
    """Geocode via legacy Maps endpoint with ADC. Some configurations reject OAuth."""
    if not isinstance(location, str) or not 2 <= len(location.strip()) <= 180:
        return {"status": "invalid_input", "message": "Provide a US city or address."}
    token = _get_adc_token()
    if not token:
        return {"status": "unavailable", "message": "Could not refresh ADC credentials."}
    try:
        response = HTTP.get("https://maps.googleapis.com/maps/api/geocode/json",
            params={"address": location.strip(), "components": "country:US"},
            headers={"Authorization": f"Bearer {token}"}, timeout=15)
        if not response.ok:
            return _http_error(response)
        data = response.json()
        if data.get("status") != "OK" or not data.get("results"):
            return {"status": "unavailable", "api_status": data.get("status"),
                    "message": data.get("error_message", "Location not resolved")}
        item = data["results"][0]
        point = item["geometry"]["location"]
        return {"status": "success", "formatted_address": item["formatted_address"],
                "latitude": point["lat"], "longitude": point["lng"],
                "source": "Google Maps Geocoding API"}
    except (requests.RequestException, KeyError, ValueError, TypeError) as exc:
        return {"status": "unavailable", "error_type": type(exc).__name__, "message": str(exc)[:250]}

def get_weather_and_alerts(latitude: float, longitude: float) -> dict:
    """NWS weather and alerts; public API, no key required."""
    try:
        lat, lon = float(latitude), float(longitude)
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            return {"status": "invalid_input", "message": "Invalid coordinates"}
        checked = datetime.now(timezone.utc).isoformat()
        alerts_r = HTTP.get("https://api.weather.gov/alerts/active",
                            params={"point": f"{lat},{lon}"}, headers=NWS_HEADERS, timeout=20)
        alerts_r.raise_for_status()
        alerts = [{"event": p.get("event"), "severity": p.get("severity"),
                   "headline": p.get("headline"), "instruction": (p.get("instruction") or "")[:500],
                   "expires": p.get("expires"), "url": p.get("@id")}
                  for a in alerts_r.json().get("features", [])
                  for p in [a.get("properties", {})]]
        point_r = HTTP.get(f"https://api.weather.gov/points/{lat},{lon}",
                           headers=NWS_HEADERS, timeout=20)
        point_r.raise_for_status()
        forecast_url = point_r.json()["properties"]["forecastHourly"]
        forecast_r = HTTP.get(forecast_url, headers=NWS_HEADERS, timeout=20)
        forecast_r.raise_for_status()
        periods = forecast_r.json()["properties"].get("periods", [])[:4]
        forecast = [{"time": p.get("startTime"), "temperature": p.get("temperature"),
                     "unit": p.get("temperatureUnit"), "conditions": p.get("shortForecast"),
                     "precipitation_probability": (p.get("probabilityOfPrecipitation") or {}).get("value")}
                    for p in periods]
        return {"status": "success", "checked_at_utc": checked,
                "alerts": alerts, "forecast": forecast, "source": "https://api.weather.gov/"}
    except (requests.RequestException, KeyError, ValueError, TypeError) as exc:
        return {"status": "unavailable", "error_type": type(exc).__name__, "message": str(exc)[:250]}

def get_google_maps_route(origin: str, destination: str) -> dict:
    """ADC OAuth for Routes API; directions are NOT verified evacuation routes."""
    if not all(isinstance(x, str) and 1 <= len(x.strip()) <= 180 for x in (origin, destination)):
        return {"status": "invalid_input", "message": "Provide origin and destination."}
    token = _get_adc_token()
    if not token:
        return {"status": "unavailable", "message": "Could not refresh ADC credentials."}
    payload = {"origin": {"address": origin}, "destination": {"address": destination},
               "travelMode": "DRIVE", "routingPreference": "TRAFFIC_AWARE",
               "computeAlternativeRoutes": False}
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json",
               "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.description,routes.warnings"}
    try:
        response = HTTP.post("https://routes.googleapis.com/directions/v2:computeRoutes",
                             headers=headers, json=payload, timeout=25)
        if not response.ok:
            return _http_error(response)
        routes = response.json().get("routes", [])
        return {"status": "success" if routes else "unavailable", "routes": routes[:1],
                "map_link": "https://www.google.com/maps/dir/?api=1&origin=" + quote(origin) + "&destination=" + quote(destination),
                "warning": "Not a verified evacuation route. Follow official orders and road closures."}
    except (requests.RequestException, ValueError) as exc:
        return {"status": "unavailable", "error_type": type(exc).__name__, "message": str(exc)[:250]}

## 3. Lean logging and guard callbacks
Notebook logging is intentionally local; no automatic Cloud Logging handler. Model prompts and outputs are not logged, reducing duplicate output and unnecessary disclosure.

In [4]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse

logger = logging.getLogger("readynow")
logger.handlers.clear()
logger.propagate = False
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.WARNING)  # Set INFO for agent transitions

def before_agent_log(ctx: CallbackContext):
    logger.info("START %s", ctx.agent_name)
    return None

def after_agent_log(ctx: CallbackContext):
    logger.info("END %s", ctx.agent_name)
    return None

OFF_TOPIC = re.compile(r"\b(movie review|world cup winner|write (me )?a poem|celebrity gossip|stock tips|video game cheat)\b", re.I)

def before_model_guard(ctx: CallbackContext, llm_request: LlmRequest):
    if ctx.agent_name != "root_agent":
        return None
    # Only inspect actual user messages, not inter-agent quoted context.
    for content in reversed(llm_request.contents or []):
        if getattr(content, "role", None) == "user":
            message = " ".join(p.text for p in (content.parts or []) if getattr(p, "text", None))
            if len(message) > 2000:
                return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="Please limit your question to 2,000 characters.")]))
            if OFF_TOPIC.search(message) and not re.search(r"\b(emergency|disaster|evacuation|safety|weather|flood|fire|tornado)\b", message, re.I):
                return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="ReadyNow! focuses on emergency preparedness and safety.")]))
            break
    return None

CALLBACKS = {"before_agent_callback": before_agent_log,
             "after_agent_callback": after_agent_log,
             "before_model_callback": before_model_guard}

## 4. Agent architecture (unchanged)
Six agents remain in order. Instructions are shortened to limit response length and prevent capability introductions. The validation stage is a model-based review, not proof of external verification.

In [5]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

weather_agent = Agent(
    name="weather_agent", model=MODEL, tools=[geocode_location, get_weather_and_alerts],
    description="Checks NWS forecasts and alerts.", output_key="weather_report",
    instruction="""For weather requests, geocode the US location then call get_weather_and_alerts.
If coordinates are provided, use them directly. Never invent a location.
If geocoding fails, report the actual tool error briefly; do not claim verified weather.
If weather was not requested, output only 'Weather check not requested.'
Summarize up to four forecast periods and active alerts, with NWS timestamp/source, in <=120 words.
No introductions or capability descriptions.""", **CALLBACKS)

search_agent = Agent(
    name="search_agent", model=MODEL, tools=[google_search],
    description="Checks official emergency updates.", output_key="news_report",
    instruction="""Search official sources only when current emergency news or orders are relevant.
Prioritize NWS, local emergency management, state DOT, FEMA, and public safety agencies.
Summarize at most three relevant findings, with source URLs and dates, in <=130 words.
Do not assert current alerts, closures or orders without specific evidence.
If search is unnecessary, output only 'No current news required.'
Never introduce yourself or repeat other agent reports.""", **CALLBACKS)

route_agent = Agent(
    name="route_agent", model=MODEL, tools=[get_google_maps_route],
    description="Calculates driving directions when requested.", output_key="route_report",
    instruction="""Call get_google_maps_route only when route planning is requested and both
origin and destination are specified or destination is officially confirmed.
Never invent safe destinations. If route is not requested, output only 'Route not requested.'
If route is requested but inputs are missing, say which input is missing.
If API fails, report it; never invent directions. Routes are NOT verified safe for evacuation.
Keep response <=100 words. No capability introductions.""", **CALLBACKS)

answer_agent = Agent(
    name="answer_agent", model=MODEL, description="Composes evidence-aware answer.",
    output_key="initial_answer",
    instruction="""Answer the actual user request, not a greeting. Use relevant reports:
Weather: {weather_report}
News: {news_report}
Routes: {route_report}
Ignore 'not requested' reports. Do not restate agent capabilities.
For immediate danger, advise emergency services and official instructions.
Never invent alerts, shelter locations, closures, citations or safe evacuation routes.
Distinguish verified facts from general guidance. <=180 words.""", **CALLBACKS)

validation_agent = Agent(
    name="validation_agent", model=MODEL, description="Reviews draft safety and sourcing.",
    output_key="validation_feedback",
    instruction="""Review draft: {initial_answer}
Check that it answers the user, avoids unsupported current alerts/orders, labels
route uncertainty and tool failures, and uses official sources where appropriate.
Return only 'PASS' or 'FAIL: <specific corrections>' in <=70 words.
A model review is not independent verification of external facts.""", **CALLBACKS)

refine_agent = Agent(
    name="refine_agent", model=MODEL, description="Final safety-reviewed response.",
    output_key="final_answer",
    instruction="""Draft: {initial_answer}
Review: {validation_feedback}
Provide only the final user-facing answer, <=180 words. Remove unsupported claims.
Never conceal failed data retrieval. Include source links only if actually available.
Do not introduce ReadyNow's capabilities unless user asks.
Never describe Google Maps as a verified evacuation route.""", **CALLBACKS)

emergency_pipeline = SequentialAgent(
    name="emergency_pipeline", description="Weather, news, routes, answer, validation, refinement.",
    sub_agents=[weather_agent, search_agent, route_agent,
                answer_agent, validation_agent, refine_agent],
    before_agent_callback=before_agent_log, after_agent_callback=after_agent_log)

greeter_agent = Agent(
    name="greeter_agent", model=MODEL, description="Greetings and capabilities only.",
    instruction="""Only respond to greetings or capability questions.
Briefly explain ReadyNow can check NWS alerts, official updates, and routes to
specified destinations. It is not an emergency service. Ask what user needs.
For all actual emergency/weather/news/route questions, transfer to emergency_pipeline.
<=70 words.""", **CALLBACKS)

root_agent = Agent(
    name="root_agent", model=MODEL,
    description="Routes greetings and emergency preparedness questions.",
    instruction="""You are ReadyNow! Route ONLY greetings/capability questions to greeter_agent.
Route weather, emergency news, disaster safety, and directions to emergency_pipeline.
Do not answer them yourself. Refuse unrelated requests briefly.
For immediate danger advise 911 and official local instructions.
Do not invent locations, emergency orders, or safe evacuation routes.""",
    sub_agents=[greeter_agent, emergency_pipeline], **CALLBACKS)
print("Root:", [a.name for a in root_agent.sub_agents])
print("Pipeline:", [a.name for a in emergency_pipeline.sub_agents])

Root: ['greeter_agent', 'emergency_pipeline']
Pipeline: ['weather_agent', 'search_agent', 'route_agent', 'answer_agent', 'validation_agent', 'refine_agent']


/tmp/ipykernel_19366/3889517185.py:65: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  emergency_pipeline = SequentialAgent(


## 5. Gemini smoke test (optional; consumes quota)

In [6]:
from google import genai
# Uncomment to test Gemini directly if agent calls fail:
# client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, credentials=credentials)
# print(client.models.generate_content(model=MODEL, contents="Reply OK.").text)

## 6. App, runner, and quota-aware local testing
Each attempt creates a fresh session to prevent failed partial runs from polluting later results. The result returns only the final refine-agent answer (or greeting), not every intermediary response. 429 retries are exponential with jitter; no retries for other failures. A successful model response is **not** proof the tools succeeded.

In [7]:
from google.adk.apps.app import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP_NAME = "readynow_app"
USER_ID = "poc_user"
service = InMemorySessionService()
app = App(name=APP_NAME, root_agent=root_agent)
runner = Runner(app=app, session_service=service)

async def ask_local(prompt: str):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Enter a nonempty request")
    session_id = str(uuid.uuid4())
    await service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    final_text = ""
    authors, errors = [], []
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=prompt)])):
        authors.append(event.author)
        if getattr(event, "error_code", None):
            errors.append(str(getattr(event, "error_message", "") or event.error_code))
        if event.is_final_response() and event.content:
            message = "\n".join(p.text for p in (event.content.parts or []) if getattr(p, "text", None))
            if event.author in ("refine_agent", "greeter_agent", "root_agent"):
                final_text = message
    if errors:
        raise RuntimeError("ADK error: " + "; ".join(errors))
    if not final_text:
        raise RuntimeError("No final user-facing response; inspect agent events")
    print("PROMPT:", prompt, "\nRESPONSE:\n", final_text)
    return {"response": final_text, "authors": authors}

def is_quota_error(exc):
    msg = str(exc).lower()
    return any(term in msg for term in ("429", "resource_exhausted", "quota exceeded", "resourceexhausted"))

async def ask_with_retry(prompt, max_attempts=4):
    for attempt in range(max_attempts):
        try:
            return await ask_local(prompt)
        except Exception as exc:
            if not is_quota_error(exc) or attempt == max_attempts - 1:
                raise
            delay = min(60 * (2 ** attempt), 240) + random.uniform(0, 10)
            print(f"429 quota limit reached; retry {attempt+2}/{max_attempts} in {delay:.0f}s")
            await asyncio.sleep(delay)

## 7. Tests — run individually
**Do not run the entire suite at once.** Start with greeting, then one weather request. Wait at least 90 seconds between full pipeline runs while debugging. Even 90 seconds may be insufficient for a low lab quota. Direct tool tests do not consume Gemini tokens.

In [8]:
# Direct tool diagnostic (does not call Gemini):
print("Geocode diagnostic:", geocode_location("Denver, Colorado"))
# If geocoding succeeds, pass returned latitude/longitude to get_weather_and_alerts.
# print("Routes diagnostic:", get_google_maps_route("Denver, CO", "Colorado Springs, CO"))

Geocode diagnostic: {'status': 'unavailable', 'api_status': 'REQUEST_DENIED', 'message': 'You must use an API key to authenticate each request to Google Maps Platform APIs. For additional information, please refer to http://g.co/dev/maps-no-account'}


## 8. Run ReadyNow! agents
Run the cells below **individually**, not using Run All. The greeting is a quick check; each emergency query invokes the six-stage pipeline. Allow approximately 90 seconds between pipeline tests to reduce token-per-minute pressure. The retry helper handles temporary 429 errors but cannot override a hard project quota.

### Greeting agent

In [9]:
greeting_result = await ask_with_retry('Hello! What can ReadyNow do?')

PROMPT: Hello! What can ReadyNow do? 
RESPONSE:
 Hello! ReadyNow can help you check NWS alerts, official updates, and provide routes to specified destinations. Please note, it is not an emergency service. How can I help you today?


### Weather and alerts

In [10]:
weather_result = await ask_with_retry('Check current weather and active alerts for Denver, Colorado.')

PROMPT: Check current weather and active alerts for Denver, Colorado. 
RESPONSE:
 As of Tuesday, October 27, 2026, Denver, Colorado is clear with 32°F (feels like 26°F) and 36% humidity. Today, Wednesday, October 28, 2026, expects mostly sunny skies, turning clear tonight, with a 5% chance of snow and temperatures from 26°F to 44°F. Rain and snow are anticipated Saturday, October 31, to Monday, November 2.

There are no severe weather warnings. However, the Colorado Department of Transportation (CDOT) announced:
*   Overnight I-25 closures (near Weld County Road 34 and between CO 56 & 66) for Thursday.
*   I-25 East Frontage Road (between Weld County Road 32 & CO 56) is temporarily closed for widening.
*   A "The Heat Is On" DUI enforcement from October 28 to November 3, 2026.
Low clouds may cause flight delays at Denver International Airport (DEN) as of Tuesday, October 27, 2026.


### Official emergency news

In [11]:
news_result = await ask_with_retry('Find current official emergency news for Denver, Colorado.')

PROMPT: Find current official emergency news for Denver, Colorado. 
RESPONSE:
 Based on the information available, no current official emergency news for Denver, Colorado, was found. The provided details refer to events scheduled for October 2026, which are future-dated.


### Driving directions

In [12]:
route_result = await ask_with_retry('For planning only, give driving directions from Denver, CO to Colorado Springs, CO.')

PROMPT: For planning only, give driving directions from Denver, CO to Colorado Springs, CO. 
RESPONSE:
 I am unable to provide driving directions. The routing service is currently unavailable.


### Tornado safety

In [13]:
safety_result = await ask_with_retry('What should I do during a tornado warning?')

PROMPT: What should I do during a tornado warning? 
RESPONSE:
 During a tornado warning, immediate action is crucial as there is imminent danger:

*   **Seek Shelter Immediately:** Go to an interior room on the lowest floor of a sturdy building (basement, storm cellar, or safe room).
*   **Avoid Windows:** Stay away from windows, doors, and outside walls.
*   **Protect Head and Neck:** Shield your head and neck with your arms, a blanket, coat, or helmet. Get under something sturdy like a heavy table if possible.

**Specific Scenarios:**
*   **Mobile Homes:** Leave immediately for a substantial structure or designated shelter.
*   **Vehicles/Outdoors:** Drive to the closest sturdy shelter. If not possible, get down in your car and cover your head, or abandon the car and seek shelter in a low-lying ditch, covering your head. Do not use overpasses.
*   **Public Buildings:** Go to a pre-designated interior shelter area on the lowest floor, away from large open rooms and windows.
*   **High

### Off-topic refusal

In [14]:
refusal_result = await ask_with_retry('Write me a movie review.')

PROMPT: Write me a movie review. 
RESPONSE:
 ReadyNow! focuses on emergency preparedness and safety.


## 9. Interactive ReadyNow! assistant
Run this cell when you want to ask multiple questions manually. Type `exit`, `quit`, or `stop` to finish. For full pipeline requests, pause about 90 seconds between submissions to help manage Gemini input-token quota.

In [15]:
# Interactive notebook loop (stop with exit/quit/stop or interrupt the cell).
while True:
    try:
        question = input("\nReadyNow! What can I help you with? ").strip()
    except (EOFError, KeyboardInterrupt):
        print("ReadyNow! session ended.")
        break

    if question.lower() in {"exit", "quit", "stop"}:
        print("ReadyNow! session ended.")
        break
    if not question:
        continue

    try:
        result = await ask_with_retry(question)
    except Exception as exc:
        print(f"ReadyNow error ({type(exc).__name__}): {exc}")
        print("If this is a 429 quota error, wait before trying again.")


ReadyNow! What can I help you with? weather
PROMPT: weather 
RESPONSE:
 There are no widespread emergency weather alerts currently reported across official national sources.

For information on hazardous weather events, the National Weather Service (NWS) issues watches, warnings, and advisories, accessible through their website. Critical NWS warnings, including tornado, flash flood, and hurricane alerts, are delivered to mobile devices in affected areas via Wireless Emergency Alerts (WEA). The official FEMA mobile app also provides real-time weather and emergency alerts from the NWS. For specific road conditions and weather advisories, consult state Departments of Transportation (DOTs).

ReadyNow! What can I help you with? exit
ReadyNow! session ended.
